# Corrected-teacher eval: HumanEval-TS

Part of plan.md step 3.2 -- same suite and harness as 1.2's baseline eval
(`02a_baseline_humaneval_ts.ipynb`), same primary/gating MultiPL-E task
(`multiple-ts`), pass@1 at temperature 0.2 / nucleus top-p 0.95 -- only the
model pointer changes, from `Qwen/Qwen3-30B-A3B-Base` to the corrected
teacher (`sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected`, 3.1). Purpose: 3.1's validation-loss
improvement (0.9331 -> 0.8842) is a proxy for "correction helped," not
proof -- this is the direct check against the metric the project actually
gates on.

Reuses 1.2's `batch_size=8` (not the harness's namesake-suggesting-bigger
default): `batch_size=32` was tried during 1.2 and hung for over an hour --
this MoE model's batched decode doesn't amortize across sequences that
route to different experts the way a dense model's would, and batching also
waits on the slowest sequence in the batch to hit a stop token.

Needs a rented GPU (1x 80GB) with the `eval` extra installed
(`pip install -e ".[eval]"`), plus Node.js + `typescript` (`tsc`).

In [1]:
import json

from bonsai.config import load_config
from bonsai.tracking import new_run, save_config_snapshot

config = load_config("../configs/qwen3-30b-a3b.toml")
model_repo_id = "sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected"
run_dir = new_run("teacher-corrected-eval")
save_config_snapshot(run_dir, config)

print(f"model: {model_repo_id}")
print(f"run dir: {run_dir}")

model: sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected
run dir: runs/20260922T001406Z-6cd438-teacher-corrected-eval


In [2]:
import os

os.environ["MODEL_REPO_ID"] = model_repo_id
os.environ["MODEL_REVISION"] = "main"
os.environ["RUN_DIR"] = str(run_dir)

In [3]:
!stdbuf -o0 -e0 accelerate launch ../tools/bigcode_eval_main.py \
    --model "$MODEL_REPO_ID" --revision "$MODEL_REVISION" --precision bf16 \
    --tasks multiple-ts \
    --max_length_generation 512 --batch_size 8 \
    --allow_code_execution --save_generations \
    --metric_output_path "$RUN_DIR/humaneval-ts.json" \
    --save_generations_path "$RUN_DIR/humaneval-ts-generations.json" \
    2>&1 | tee /tmp/live_humaneval_ts_corrected.log

assert _exit_code == 0, "accelerate launch failed for multiple-ts"  # noqa: F821 -- set by the ! shell magic above at runtime

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.


Selected Tasks: ['multiple-ts']
Loading model in bf16
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files:  50%|█████     | 1/2 [00:28<00:28, 28.98s/it]

Fetching 2 files: 100%|██████████| 2/2 [01:31<00:00, 45.55s/it]


Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

Loading weights:   1%|          | 4/531 [00:00<00:58,  8.96it/s]

... (415 outputs left out)

 99%|█████████▊| 157/159 [02:17<00:01,  1.59it/s]

 99%|█████████▉| 158/159 [02:17<00:00,  1.88it/s]

100%|██████████| 159/159 [02:18<00:00,  1.15it/s]


{
  "multiple-ts": {
    "pass@1": 0.559748427672956
  },
  "config": {
    "prefix": "",
    "do_sample": true,
    "temperature": 0.2,
    "top_k": 0,
    "top_p": 0.95,
    "n_samples": 1,
    "eos": "<|endoftext|>",
    "seed": 0,
    "model": "sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected",
    "modeltype": "causal",
    "peft_model": null,
    "revision": "main",
    "use_auth_token": false,
    "trust_remote_code": false,
    "tasks": "multiple-ts",
    "instruction_tokens": null,
    "batch_size": 8,
    "max_length_generation": 512,
    "precision": "bf16",
    "load_in_8bit": false,
    "load_in_4bit": false,
    "left_padding": false,
    "limit": null,
    "limit_start": 0,
    "save_every_k_tasks": -1,
    "postprocess": true,
    "allow_code_execution": true,
    "generation_only": false,
    "load_generations_path": null,
    "load_data_path": null,
    "metric_output_path": "runs/20260922T001406Z-6cd438-teacher-corrected-eval/humaneval-ts.json",
    "save_generat

In [4]:
humaneval_ts = json.loads((run_dir / "humaneval-ts.json").read_text())
print("humaneval-ts (corrected teacher):", humaneval_ts)

humaneval-ts (corrected teacher): {'multiple-ts': {'pass@1': 0.559748427672956}, 'config': {'prefix': '', 'do_sample': True, 'temperature': 0.2, 'top_k': 0, 'top_p': 0.95, 'n_samples': 1, 'eos': '<|endoftext|>', 'seed': 0, 'model': 'sartajbhuvaji/bonsai-qwen3-30b-a3b-teacher-corrected', 'modeltype': 'causal', 'peft_model': None, 'revision': 'main', 'use_auth_token': False, 'trust_remote_code': False, 'tasks': 'multiple-ts', 'instruction_tokens': None, 'batch_size': 8, 'max_length_generation': 512, 'precision': 'bf16', 'load_in_8bit': False, 'load_in_4bit': False, 'left_padding': False, 'limit': None, 'limit_start': 0, 'save_every_k_tasks': -1, 'postprocess': True, 'allow_code_execution': True, 'generation_only': False, 'load_generations_path': None, 'load_data_path': None, 'metric_output_path': 'runs/20260922T001406Z-6cd438-teacher-corrected-eval/humaneval-ts.json', 'save_generations': True, 'load_generations_intermediate_paths': None, 'save_generations_path': 'runs/20260922T001406Z-6c